# Extra cold-target split seeds

Phase 1 froze every split at seed 4001, so the standard deviations reported in
Phase 2 measure training noise only. They say nothing about how much the
warm-to-cold gap depends on *which genes happened to land in test*.

This notebook regenerates `split_cold_target` under three further split seeds
using Phase 1's grouping rule unchanged, and writes them to a separate file so
`data/processed/{ds}_pairs.csv.gz` stays byte-identical to Phase 1's export.

Split seeds are not training seeds. A split seed chooses the partition; a
training seed chooses initialisation and batch order. They are varied
independently.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

PROCESSED = Path("../data/processed") if Path("../data/processed").exists() else Path("data/processed")
PARTITIONS = ("train", "val", "test")
SPLIT_SEEDS = (4002, 4003, 4004)
GZIP = {"method": "gzip", "mtime": 0}


def cold_target_split(df, entity_targets, seed):
    """Assign whole target groups near cumulative 70/15/15 pair boundaries.

    This is Phase 1's rule verbatim except that the seed is a parameter, so the
    partitions stay comparable with `split_cold_target` in the pairs table.

    Args:
        df: Content-keyed pair table.
        entity_targets: Target entities carrying target_group.
        seed: Split seed selecting the group permutation.

    Returns:
        Partition labels aligned to the pair table's index.
    """
    group_of = entity_targets.set_index("target_id").target_group
    group_keys = df.target_id.map(group_of)
    sizes = group_keys.value_counts(sort=False)
    order = np.random.default_rng(seed).permutation(len(sizes))
    assignments, cumulative = {}, 0
    boundaries = np.array([0.70, 0.85]) * len(df)
    for i in order:
        group, count = sizes.index[i], int(sizes.iloc[i])
        midpoint = cumulative + count / 2
        assignments[group] = PARTITIONS[np.searchsorted(boundaries, midpoint, side="right")]
        cumulative += count
    return group_keys.map(assignments)

In [2]:
summary = []
for ds in ("davis", "kiba"):
    pairs = pd.read_csv(PROCESSED / f"{ds}_pairs.csv.gz", dtype={"ligand_id": str, "target_id": str})
    targets = pd.read_csv(PROCESSED / f"{ds}_targets.csv.gz", dtype=str)
    extra = pairs[["ligand_id", "target_id"]].copy()
    group_of = targets.set_index("target_id").target_group

    for seed in SPLIT_SEEDS:
        column = f"split_cold_target_s{seed}"
        extra[column] = cold_target_split(pairs, targets, seed)
        assert extra[column].isin(PARTITIONS).all(), f"{column} has unexpected labels"

        # A gene's wild type and all its mutants must stay in one partition.
        target_sets, group_sets = [], []
        for partition in PARTITIONS:
            subset = extra[extra[column] == partition]
            target_sets.append(set(subset.target_id))
            group_sets.append(set(subset.target_id.map(group_of)))
            assert len(subset) > 0, f"{ds}/{column}: empty {partition}"
        for i in range(3):
            for j in range(i + 1, 3):
                assert target_sets[i].isdisjoint(target_sets[j]), f"{ds}/{column}: targets shared"
                assert group_sets[i].isdisjoint(group_sets[j]), f"{ds}/{column}: groups shared"

        fractions = [len(extra[extra[column] == p]) / len(extra) for p in PARTITIONS]
        summary.append({"dataset": ds, "split_seed": seed,
                        **{f"frac_{p}": round(f, 4) for p, f in zip(PARTITIONS, fractions)},
                        **{f"targets_{p}": len(s) for p, s in zip(PARTITIONS, target_sets)}})
        assert 0.60 <= fractions[0] <= 0.80, f"{ds}/{column}: train fraction {fractions[0]:.3f} off target"

    extra.to_csv(PROCESSED / f"{ds}_extra_splits.csv.gz", index=False, compression=GZIP)

print(pd.DataFrame(summary).to_string(index=False))

dataset  split_seed  frac_train  frac_val  frac_test  targets_train  targets_val  targets_test
  davis        4002      0.6998    0.1513     0.1489            296           64            63
  davis        4003      0.6998    0.1513     0.1489            296           64            63
  davis        4004      0.6998    0.1513     0.1489            296           64            63
   kiba        4002      0.7021    0.1477     0.1502            158           34            37
   kiba        4003      0.6990    0.1508     0.1502            163           30            36
   kiba        4004      0.6990    0.1502     0.1508            165           29            35


In [3]:
# Reload and prove the new columns align with the pairs table row for row,
# and that each split seed produces a genuinely different partition.
for ds in ("davis", "kiba"):
    pairs = pd.read_csv(PROCESSED / f"{ds}_pairs.csv.gz", dtype={"ligand_id": str, "target_id": str})
    extra = pd.read_csv(PROCESSED / f"{ds}_extra_splits.csv.gz", dtype={"ligand_id": str, "target_id": str})
    assert len(extra) == len(pairs), f"{ds}: row count differs"
    assert (extra.ligand_id == pairs.ligand_id).all() and (extra.target_id == pairs.target_id).all(), \
        f"{ds}: row order differs from the pairs table"
    columns = [f"split_cold_target_s{s}" for s in SPLIT_SEEDS]
    for a in [*columns, "split_cold_target"]:
        for b in columns:
            if a == b:
                continue
            agreement = (pairs["split_cold_target"] if a == "split_cold_target" else extra[a]).eq(extra[b]).mean()
            assert agreement < 0.99, f"{ds}: {a} and {b} agree on {agreement:.3f} of pairs; not a new split"
    test_targets = {c: set(extra.loc[extra[c] == "test", "target_id"]) for c in columns}
    base_test = set(pairs.loc[pairs.split_cold_target == "test", "target_id"])
    print(f"{ds}: test-target overlap with the original split:",
          {c: len(test_targets[c] & base_test) for c in columns},
          f"(original test targets: {len(base_test)})")
print("All extra-split checks passed.")

davis: test-target overlap with the original split: {'split_cold_target_s4002': 7, 'split_cold_target_s4003': 9, 'split_cold_target_s4004': 14} (original test targets: 63)


kiba: test-target overlap with the original split: {'split_cold_target_s4002': 6, 'split_cold_target_s4003': 3, 'split_cold_target_s4004': 6} (original test targets: 30)
All extra-split checks passed.
